# 구매요청 중단 예측 학습 노트북

이 노트북은 `PURCHASE_REQUEST_ITEMS.csv`를 사용해 `중단`(0/1) 분류 모델을 학습합니다.

진행 순서
- 데이터 로드
- 타깃 생성
- 학습 컬럼 선택
- 전처리 + 모델 학습
- 성능 평가
- 모델 저장

In [1]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

def cast_to_str(X):
    return X.astype(str)

In [2]:
ROOT = Path("..").resolve()
CSV_PATH = ROOT / "data" / "processed" / "PURCHASE_REQUEST_ITEMS.csv"
MODEL_DIR = ROOT / "modellearn" / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("CSV:", CSV_PATH)
print("exists:", CSV_PATH.exists())
print("MODEL_DIR:", MODEL_DIR)

CSV: C:\Workspaces\sllm_texttosql_ml_rag_ortools_simulation_langgraph\data\processed\PURCHASE_REQUEST_ITEMS.csv
exists: True
MODEL_DIR: C:\Workspaces\sllm_texttosql_ml_rag_ortools_simulation_langgraph\modellearn\artifacts


## 1. 데이터 로드

In [3]:
# 인코딩 이슈 시: encoding="cp949"
df = pd.read_csv(CSV_PATH, encoding="utf-8", low_memory=False)
print("shape:", df.shape)
df.head(2)

shape: (26472, 51)


,선택,확정,중단,중단일,중단사유,중단처리자,요청일,요청부서,구매요청번호,요청담당자,품번,품명,품목약명,품목영문명,구매요청수량,품목영문약명,규격,품목분류1,품목분류2,품목분류3,품목분류4,구매단위,품의진행수량,발주진행수량,요청납기일,내/외자,창고,진행상태,비고,품목자산분류,Maker,수주거래처,수주번호,비고2,PONO,품목담당자,거래처,구매요청구분,발주예정일,생산계획번호,자재소요번호,원천조회,진행조회,등록일시,조달일수,구매발주(수입Order)번호,화학물질오류여부,가용재고보유여부,중단요청,직배송,직배송주소
0,NaN,True,False,NaN,NaN,NaN,2020-04-01,남부영업팀(2020),202004010001,김준수,A11070,"[1분기 WSP 행사,~3/31, 소비자 15%,딜러(소가기준) 20%할인] ALE...",ALEXA FLUOR 488 F(AB) 250 UL,ALEXA FLUOR 488 F(AB) 250 UL,1,ALEXA FLUOR 488 F(AB) 250 UL,WSP행사,Thermo_BID,시약류,PRO,NaN,EA,1,1,2020-04-29,내수,기본창고(평택물류),완료,NaN,상품,Life Technologies,랩솔루션(부산)(e),2.020040e+12,NaN,NaN,NaN,써모피셔사이언티픽솔루션스 유한회사,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,YLTK-200401B,NaN,False,False,False,NaN
1,NaN,True,False,NaN,NaN,NaN,2020-04-01,영업1부(2020),202004010002,송주미,5020-03945,"Inertsil ODS-4, 5um, Columns 4.6 x 150mm","Inertsil ODS-4, 5um, Columns 4.6 x 150mm","Inertsil ODS-4, 5um, Columns 4.6 x 150mm",1,NaN,NaN,GL Sciences,소모품,HPLC Columns,NaN,PK,1,1,2020-04-29,내수,기본창고(평택물류),완료,직배송요청\n경기 용인시 기흥구 탑실로35번길 25 애드파마 \n조성조 010-65...,상품,GL Sciences,애드파마,2.020033e+12,NaN,NaN,NaN,국제씨에스(법인)(e),NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,*YIJ-200401A,NaN,False,False,False,NaN


## 2. 타깃(중단) 0/1 생성

In [4]:
if "중단" not in df.columns:
    raise KeyError("컬럼 '중단'이 없습니다.")

def to_binary_stop(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (bool, np.bool_)):
        return int(bool(x))
    s = str(x).strip().lower()
    if s in ("true", "1", "y", "yes", "중단"):
        return 1
    if s in ("false", "0", "n", "no", ""):
        return 0
    return np.nan

y = df["중단"].map(to_binary_stop)
print("target raw counts")
print(y.value_counts(dropna=False))

# 학습에는 타깃이 있는 행만 사용
work = df.copy()
work["target"] = y
work = work.dropna(subset=["target"]).copy()
work["target"] = work["target"].astype(int)
print("usable rows:", len(work))
print("positive rate:", round(work["target"].mean(), 4))

target raw counts
중단
0    25641
1      831
Name: count, dtype: int64
usable rows: 26472
positive rate: 0.0314


## 3. 학습 컬럼 선택 (누수 방지)

In [5]:
# 기본 제외: 타깃/사후 정보/식별자/누수 의심 컬럼
leakage_cols = [
    "중단", "target", "중단일", "중단사유", "중단처리자", "중단요청",
    "확정", "진행상태", "진행조회", "원천조회",  # 결과 이후 상태를 담을 가능성
    "구매요청번호", "자재소요번호", "생산계획번호", "PONO",
]

feature_cols = [c for c in work.columns if c not in leakage_cols]
X = work[feature_cols].copy()
y = work["target"].copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print("features:", X.shape[1])
print("num:", len(num_cols), "cat:", len(cat_cols))
print("dropped:", leakage_cols)
print("sample feature columns:", feature_cols[:15])

features: 38
num: 11 cat: 27
dropped: ['중단', 'target', '중단일', '중단사유', '중단처리자', '중단요청', '확정', '진행상태', '진행조회', '원천조회', '구매요청번호', '자재소요번호', '생산계획번호', 'PONO']
sample feature columns: ['선택', '요청일', '요청부서', '요청담당자', '품번', '품명', '품목약명', '품목영문명', '구매요청수량', '품목영문약명', '규격', '품목분류1', '품목분류2', '품목분류3', '품목분류4']


## 4. 분할 (시간 우선, 없으면 Stratified)

In [6]:
if "요청일" in work.columns:
    dt = pd.to_datetime(work["요청일"], errors="coerce")
    valid_idx = dt.notna()
    X2 = X.loc[valid_idx].copy()
    y2 = y.loc[valid_idx].copy()
    dt2 = dt.loc[valid_idx]

    cutoff = dt2.quantile(0.8)
    train_mask = dt2 <= cutoff

    X_train, X_test = X2.loc[train_mask], X2.loc[~train_mask]
    y_train, y_test = y2.loc[train_mask], y2.loc[~train_mask]
    split_mode = f"time split (cutoff={cutoff.date()})"
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
    split_mode = "stratified random split"

print("split mode:", split_mode)
print("train:", X_train.shape, "test:", X_test.shape)
print("train pos rate:", round(y_train.mean(), 4), "test pos rate:", round(y_test.mean(), 4))

split mode: time split (cutoff=2020-06-15)
train: (21549, 38) test: (4923, 38)
train pos rate: 0.0314 test pos rate: 0.0315


## 5. 전처리 + 모델 정의

In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# 범주형 컬럼은 dtype이 섞여 있어도 문자열로 통일해서 처리
categorical_transformer = Pipeline(
    steps=[
        ("to_str", FunctionTransformer(cast_to_str, feature_names_out="one-to-one", validate=False)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

models = {
    "logreg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "rf": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=10,
        min_samples_split=20,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
}

## 6. 학습 및 평가

In [8]:
results = []
fitted = {}

for name, model in models.items():
    clf = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    clf.fit(X_train, y_train)

    # train 성능(과적합 점검)
    train_prob = clf.predict_proba(X_train)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)
    train_f1 = f1_score(y_train, train_pred, zero_division=0)
    train_auc = roc_auc_score(y_train, train_prob)

    # test 성능
    test_prob = clf.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)
    test_auc = roc_auc_score(y_test, test_prob)

    results.append({
        "model": name,
        "train_f1": train_f1,
        "test_f1": test_f1,
        "f1_gap": train_f1 - test_f1,
        "test_accuracy": test_acc,
        "train_roc_auc": train_auc,
        "test_roc_auc": test_auc,
        "auc_gap": train_auc - test_auc,
    })
    fitted[name] = clf

result_df = pd.DataFrame(results).sort_values(["test_f1", "test_roc_auc"], ascending=False)
result_df

,model,train_f1,test_f1,f1_gap,test_accuracy,train_roc_auc,test_roc_auc,auc_gap
1,rf,0.757447,0.671605,0.085842,0.972984,0.987600,0.986248,0.001352
0,logreg,0.060832,0.061048,-0.000215,0.031485,0.511495,0.541463,-0.029968


In [9]:
best_name = result_df.iloc[0]["model"]
best_model = fitted[best_name]

best_prob = best_model.predict_proba(X_test)[:, 1]
best_pred = (best_prob >= 0.5).astype(int)

print("best model:", best_name)
print("\n[Overfit check]")
row = result_df[result_df["model"] == best_name].iloc[0]
print(f"train_f1={row['train_f1']:.4f}, test_f1={row['test_f1']:.4f}, gap={row['f1_gap']:.4f}")
print(f"train_auc={row['train_roc_auc']:.4f}, test_auc={row['test_roc_auc']:.4f}, gap={row['auc_gap']:.4f}")
print("\n[Confusion Matrix]")
print(confusion_matrix(y_test, best_pred))
print("\n[Classification Report]")
print(classification_report(y_test, best_pred, digits=4, zero_division=0))

# 클래스 1(중단) 중심 요약 지표
p1 = precision_score(y_test, best_pred, pos_label=1, zero_division=0)
r1 = recall_score(y_test, best_pred, pos_label=1, zero_division=0)
f1_1 = f1_score(y_test, best_pred, pos_label=1, zero_division=0)
pr_auc = average_precision_score(y_test, best_prob)
base_rate = y_test.mean()

print("\n[Class-1 Summary]")
print(f"precision_1={p1:.4f}, recall_1={r1:.4f}, f1_1={f1_1:.4f}")
print(f"pr_auc={pr_auc:.4f}, positive_rate(test)={base_rate:.4f}")

best model: rf

[Overfit check]
train_f1=0.7574, test_f1=0.6716, gap=0.0858
train_auc=0.9876, test_auc=0.9862, gap=0.0014

[Confusion Matrix]
[[4654  114]
 [  19  136]]

[Classification Report]
              precision    recall  f1-score   support

           0     0.9959    0.9761    0.9859      4768
           1     0.5440    0.8774    0.6716       155

    accuracy                         0.9730      4923
   macro avg     0.7700    0.9268    0.8288      4923
weighted avg     0.9817    0.9730    0.9760      4923


[Class-1 Summary]
precision_1=0.5440, recall_1=0.8774, f1_1=0.6716
pr_auc=0.8811, positive_rate(test)=0.0315


## 7. 임계값 간단 점검

In [10]:
prec, rec, thr = precision_recall_curve(y_test, best_prob)
thr = np.append(thr, 1.0)
scan = pd.DataFrame({"threshold": thr, "precision": prec, "recall": rec})

# recall 0.6 이상 구간 중 precision이 가장 높은 지점
cand = scan[scan["recall"] >= 0.6].sort_values("precision", ascending=False)
scan.head(10), cand.head(5)

(   threshold  precision  recall
 0   0.285604   0.031485     1.0
 1   0.287655   0.031510     1.0
 2   0.287908   0.031568     1.0
 3   0.288344   0.031575     1.0
 4   0.289960   0.031588     1.0
 5   0.290175   0.031600     1.0
 6   0.290347   0.031607     1.0
 7   0.290396   0.031620     1.0
 8   0.291185   0.031652     1.0
 9   0.291349   0.031665     1.0,
       threshold  precision    recall
 3556   0.594031        1.0  0.625806
 3559   0.597150        1.0  0.600000
 3545   0.578334        1.0  0.696774
 3558   0.596991        1.0  0.606452
 3549   0.583596        1.0  0.670968)

## 8. 모델 저장

In [11]:
artifact_path = MODEL_DIR / f"stop_classifier_{best_name}.joblib"
joblib.dump(best_model, artifact_path)
print("saved:", artifact_path)

saved: C:\Workspaces\sllm_texttosql_ml_rag_ortools_simulation_langgraph\modellearn\artifacts\stop_classifier_rf.joblib


## 9. 결과 메모

- 누수 의심 컬럼(`확정`, `진행상태`, `중단사유` 등)을 제외하고 학습했습니다.
- 가능하면 `요청일` 기준 시간 분할로 평가합니다.
- 모델별 train/test 점수와 gap을 함께 확인해 과적합 여부를 점검합니다.
- 클래스1(중단) 기준 `precision/recall/f1`, `PR-AUC`를 함께 확인합니다.
- 운영 기준에 맞춰 threshold를 조정해 재현율/정밀도를 선택합니다.